In [1]:
!git clone https://github.com/zachshotwell/AI-FinalProject-GroupSCR
%cd AI-FinalProject-GroupSCR
%cd RunoffForecastingProject
!ls

Cloning into 'AI-FinalProject-GroupSCR'...
remote: Enumerating objects: 2086, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 2086 (delta 5), reused 4 (delta 2), pack-reused 2071 (from 2)
Receiving objects: 100% (2086/2086), 58.29 MiB | 20.52 MiB/s, done.
Resolving deltas: 100% (1664/1664), done.
Updating files: 100% (2061/2061), done.
/content/AI-FinalProject-GroupSCR
/content/AI-FinalProject-GroupSCR/RunoffForecastingProject
 data
'Improved runoff forecasting performance through error predictions using a deep-learning approach.pdf'
 README.md
 RunoffForcastingProject.pdf
 RunoffForecasting.ipynb


In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

tf.keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
"""
RunoffForecasting data reconnaissance.

Opens one USGS observation file and one monthly NWM forecast file from
each of the two station folders and prints enough information to decide:
  - long vs wide forecast format (one row per (issue, lead) vs 18 lead-time cols)
  - column names and dtypes
  - date/time column and its range
  - inferred sampling cadence
  - likely streamflow units (cfs vs cms based on magnitude)

Edit DATA_ROOT below to match wherever the data lives in your runtime:
  - local clone:  Path("RunoffForecastingProject")
  - Drive mount:  Path("/content/drive/MyDrive/RunoffForecastingProject")
  - GitHub clone: Path("/content/RunoffForecastingProject")
"""

from pathlib import Path
import pandas as pd

DATA_ROOT = Path("data")

# NWM reach (folder name)  ->  USGS gauge it pairs with
STATIONS = {
    "20380357": "09520500",
    "21609641": "11266500",
}


def find_files(folder: Path):
    """Split the CSVs in one station folder into (usgs_file, nwm_files)."""
    csvs = sorted(folder.glob("*.csv"))
    usgs = [f for f in csvs if "Strt" in f.name and "EndAt" in f.name]
    nwm = [f for f in csvs if f.name.startswith("streamflow_")]
    if len(usgs) != 1:
        raise ValueError(
            f"Expected exactly 1 USGS observation file in {folder}, "
            f"found {len(usgs)}: {[f.name for f in usgs]}"
        )
    return usgs[0], nwm


def find_datetime_col(df: pd.DataFrame):
    """Return (col_name, parsed_series) for the first parseable datetime col."""
    for c in df.columns:
        s = df[c]
        if s.dtype == "object" or "date" in c.lower() or "time" in c.lower():
            try:
                parsed = pd.to_datetime(s, errors="raise")
                return c, parsed
            except Exception:
                continue
    return None, None


def report(df: pd.DataFrame, label: str, n: int = 5) -> None:
    print(f"\n--- {label} ---")
    print(f"shape:   {df.shape}")
    print(f"columns: {list(df.columns)}")
    print("\ndtypes:")
    print(df.dtypes.to_string())

    print(f"\nhead({n}):")
    print(df.head(n).to_string())
    print(f"\ntail({n}):")
    print(df.tail(n).to_string())

    col, parsed = find_datetime_col(df)
    if col is not None:
        diffs = parsed.sort_values().diff().dropna()
        print(f"\ndatetime column: '{col}'")
        print(f"  range:        {parsed.min()}  ->  {parsed.max()}")
        print(f"  n unique:     {parsed.nunique()}")
        print(f"  median delta: {diffs.median()}")
    else:
        print("\nNo obvious datetime column detected -- inspect manually.")

    # Quick units sanity check on numeric columns
    num = df.select_dtypes(include="number")
    if not num.empty:
        print("\nnumeric column summary (look for streamflow magnitude):")
        print(num.describe().T[["min", "50%", "max"]].to_string())
        max_val = num.max(numeric_only=True).max()
        if max_val > 1000:
            print(f"  -> max value {max_val:.1f} is large; likely cfs.")
        elif max_val > 0:
            print(f"  -> max value {max_val:.1f} is modest; likely cms.")


def classify_format(df: pd.DataFrame) -> str:
    n = df.shape[1]
    if 18 <= n <= 25:
        return f"likely WIDE (one row per issue time, ~18 lead-time columns) -- {n} cols"
    if n <= 5:
        return f"likely LONG ((issue_time, lead_time, value) layout) -- {n} cols"
    return f"ambiguous -- {n} cols, inspect manually"


def main() -> None:
    for nwm_id, usgs_id in STATIONS.items():
        folder = DATA_ROOT / nwm_id
        bar = "=" * 72
        print(f"\n{bar}\nStation folder: {folder}   (paired USGS gauge: {usgs_id})\n{bar}")

        usgs_file, nwm_files = find_files(folder)
        print(f"USGS file:  {usgs_file.name}")
        print(f"NWM files:  {len(nwm_files)} found")
        if nwm_files:
            print(f"  first: {nwm_files[0].name}")
            print(f"  last:  {nwm_files[-1].name}")

        usgs_df = pd.read_csv(usgs_file)
        report(usgs_df, f"USGS observations -- {usgs_id}")

        if nwm_files:
            nwm_df = pd.read_csv(nwm_files[0])
            report(nwm_df, f"NWM forecast -- {nwm_files[0].name}")
            print(f"\nformat hint: {classify_format(nwm_df)}")


if __name__ == "__main__":
    main()



Station folder: data/20380357   (paired USGS gauge: 09520500)
USGS file:  09520500_Strt_2021-04-20_EndAt_2023-04-21.csv
NWM files:  25 found
  first: streamflow_20380357_202104.csv
  last:  streamflow_20380357_202304.csv

--- USGS observations -- 09520500 ---
shape:   (70089, 3)
columns: ['DateTime', 'USGSFlowValue', 'USGS_GageID']

dtypes:
DateTime          object
USGSFlowValue    float64
USGS_GageID       object

head(5):
                    DateTime  USGSFlowValue USGS_GageID
0  2021-04-20 07:00:00+00:00           0.18           A
1  2021-04-20 07:15:00+00:00           0.18           A
2  2021-04-20 07:30:00+00:00           0.19           A
3  2021-04-20 07:45:00+00:00           0.19           A
4  2021-04-20 08:00:00+00:00           0.19           A

tail(5):
                        DateTime  USGSFlowValue USGS_GageID
70084  2023-04-22 05:45:00+00:00           0.16           A
70085  2023-04-22 06:00:00+00:00           0.15           A
70086  2023-04-22 06:15:00+00:00           0.

/tmp/ipykernel_12371/1567078175.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(s, errors="raise")
/tmp/ipykernel_12371/1567078175.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(s, errors="raise")
/tmp/ipykernel_12371/1567078175.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(s, errors="raise")
/tmp/ipykernel_12371/1567078175.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected

In [4]:
"""
RunoffForecasting tidy loader.

For each station folder, parses the 25 monthly NWM forecast CSVs and the
single USGS observation CSV, aligns them on the hourly grid, and returns a
tidy dataframe with columns:

    valid_time   timezone-aware UTC timestamp the forecast/obs is FOR
    init_time    timezone-aware UTC timestamp the forecast was ISSUED at
    lead_h       integer lead time in hours (= valid_time - init_time), 1..18
    nwm_q        NWM forecast streamflow at valid_time, issued at init_time
    usgs_q       USGS observed streamflow at valid_time
    error        usgs_q - nwm_q  (positive = NWM underestimates)

This is the canonical input for the windowing / training pipeline.
Stations are kept separate (different basins, very different magnitudes).

USAGE:
    from runoff_loader import build_station_df, STATIONS
    df_small = build_station_df("data/20380357")
    df_large = build_station_df("data/21609641")

UNIT NOTE:
    NWM v2.1 streamflow_value is m^3/s (cms). USGS NWIS extracts can be
    cfs or cms depending on how they were pulled. If your obs come back in
    cfs, pass usgs_unit_factor=0.0283168 to convert to cms before joining.
"""

from __future__ import annotations
from pathlib import Path
import pandas as pd

# NWM uses an underscore between date and time -- explicit format avoids
# the dateutil row-by-row fallback which is ~50x slower on 300k+ rows.
NWM_DT_FMT = "%Y-%m-%d_%H:%M:%S"

STATIONS = {
    "20380357": "09520500",  # small headwater stream
    "21609641": "11266500",  # larger river
}


# ---------- USGS ----------------------------------------------------------- #

def load_usgs(usgs_file: Path, unit_factor: float = 1.0) -> pd.Series:
    """
    Load a USGS observation CSV and return an hourly Series of streamflow
    indexed by timezone-aware UTC timestamp.

    The raw extract is at 15-minute cadence; we subsample at the top of each
    hour (minute == 0) to match NWM's hourly grid, rather than averaging --
    NWM forecasts represent instantaneous values, so we do the same for obs.
    """
    df = pd.read_csv(usgs_file, parse_dates=["DateTime"])
    df = df[["DateTime", "USGSFlowValue"]].rename(
        columns={"DateTime": "valid_time", "USGSFlowValue": "usgs_q"}
    )
    df["valid_time"] = pd.to_datetime(df["valid_time"], utc=True)
    df = df.set_index("valid_time").sort_index()

    # asfreq picks the value at exactly each hour mark; missing -> NaN.
    hourly = df["usgs_q"].asfreq("h")
    return hourly * unit_factor


# ---------- NWM ------------------------------------------------------------ #

def _load_one_nwm_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["init_time"] = pd.to_datetime(
        df["model_initialization_time"], format=NWM_DT_FMT, utc=True
    )
    df["valid_time"] = pd.to_datetime(
        df["model_output_valid_time"], format=NWM_DT_FMT, utc=True
    )
    df["lead_h"] = (
        (df["valid_time"] - df["init_time"]).dt.total_seconds() // 3600
    ).astype(int)
    df = df.rename(columns={"streamflow_value": "nwm_q"})
    return df[["init_time", "valid_time", "lead_h", "nwm_q"]]


def load_nwm_station(station_folder: Path) -> pd.DataFrame:
    """Concatenate all 25 monthly NWM files for one station, dedupe overlaps."""
    files = sorted(station_folder.glob("streamflow_*.csv"))
    if not files:
        raise FileNotFoundError(f"No NWM files found in {station_folder}")
    parts = [_load_one_nwm_file(f) for f in files]
    nwm = pd.concat(parts, ignore_index=True)
    # If two consecutive monthly files overlap at the boundary, the same
    # (init_time, lead_h) pair will appear twice -- keep the first.
    nwm = nwm.drop_duplicates(subset=["init_time", "lead_h"], keep="first")
    nwm = nwm.sort_values(["init_time", "lead_h"]).reset_index(drop=True)
    return nwm


# ---------- Join ----------------------------------------------------------- #

def build_station_df(
    station_folder: str | Path,
    usgs_unit_factor: float = 1.0,
) -> pd.DataFrame:
    """
    Build the tidy [valid_time, init_time, lead_h, nwm_q, usgs_q, error] frame
    for one station.

    Args:
        station_folder: path to e.g. "data/20380357"
        usgs_unit_factor: multiplier applied to USGS values. Use 0.0283168
            if USGS is in cfs and you want everything in cms.
    """
    station_folder = Path(station_folder)

    # find USGS file by naming pattern
    usgs_files = [
        f for f in station_folder.glob("*.csv")
        if "Strt" in f.name and "EndAt" in f.name
    ]
    if len(usgs_files) != 1:
        raise ValueError(
            f"Expected 1 USGS file in {station_folder}, found "
            f"{[f.name for f in usgs_files]}"
        )

    usgs_hourly = load_usgs(usgs_files[0], unit_factor=usgs_unit_factor)
    nwm = load_nwm_station(station_folder)

    # left-join USGS onto NWM by valid_time. Missing obs -> NaN in usgs_q.
    df = nwm.merge(
        usgs_hourly.rename("usgs_q"),
        left_on="valid_time",
        right_index=True,
        how="left",
    )
    df["error"] = df["usgs_q"] - df["nwm_q"]

    return df[["valid_time", "init_time", "lead_h", "nwm_q", "usgs_q", "error"]]


# ---------- Reporting ------------------------------------------------------ #

def summarize(df: pd.DataFrame, label: str) -> None:
    print(f"\n{'='*72}\n{label}\n{'='*72}")
    print(f"rows:           {len(df):,}")
    print(f"unique inits:   {df['init_time'].nunique():,}")
    print(f"lead_h range:   {df['lead_h'].min()}..{df['lead_h'].max()}")
    print(f"valid_time:     {df['valid_time'].min()}  ->  {df['valid_time'].max()}")
    print(f"missing usgs_q: {df['usgs_q'].isna().sum():,} "
          f"({df['usgs_q'].isna().mean():.1%})")

    # NWM-vs-obs sanity at lead 1
    lead1 = df[df["lead_h"] == 1].dropna()
    if not lead1.empty:
        print(f"\nat lead_h=1 (n={len(lead1):,} aligned rows):")
        print(f"  nwm_q   median={lead1['nwm_q'].median():7.3f}  max={lead1['nwm_q'].max():8.2f}")
        print(f"  usgs_q  median={lead1['usgs_q'].median():7.3f}  max={lead1['usgs_q'].max():8.2f}")
        print(f"  error   median={lead1['error'].median():+7.3f}  mean={lead1['error'].mean():+7.3f}")
        ratio = lead1["nwm_q"].median() / max(lead1["usgs_q"].median(), 1e-9)
        print(f"  nwm/usgs median ratio: {ratio:.2f}  "
              f"(>>1 or <<1 suggests a unit mismatch)")


def main() -> None:
    data_root = Path("data")
    for nwm_id, usgs_id in STATIONS.items():
        df = build_station_df(data_root / nwm_id)
        summarize(df, f"Station {nwm_id} (USGS {usgs_id})")
        # save for downstream use
        out = data_root / nwm_id / f"tidy_{nwm_id}.parquet"
        df.to_parquet(out)
        print(f"\nwrote: {out}")


if __name__ == "__main__":
    main()



Station 20380357 (USGS 09520500)
rows:           315,789
unique inits:   17,544
lead_h range:   1..18
valid_time:     2021-04-21 01:00:00+00:00  ->  2023-04-22 17:00:00+00:00
missing usgs_q: 1,164 (0.4%)

at lead_h=1 (n=17,483 aligned rows):
  nwm_q   median=  3.930  max=  207.66
  usgs_q  median=  0.240  max=    4.08
  error   median= -3.700  mean= -4.840
  nwm/usgs median ratio: 16.37  (>>1 or <<1 suggests a unit mismatch)

wrote: data/20380357/tidy_20380357.parquet

Station 21609641 (USGS 11266500)
rows:           315,789
unique inits:   17,544
lead_h range:   1..18
valid_time:     2021-04-21 01:00:00+00:00  ->  2023-04-22 17:00:00+00:00
missing usgs_q: 18,372 (5.8%)

at lead_h=1 (n=16,527 aligned rows):
  nwm_q   median=  6.300  max=  129.28
  usgs_q  median=  7.080  max=  133.66
  error   median= +0.030  mean= +0.282
  nwm/usgs median ratio: 0.89  (>>1 or <<1 suggests a unit mismatch)

wrote: data/21609641/tidy_21609641.parquet


In [5]:
"""
Train/val/test split for the tidy runoff parquets.

Pattern follows Géron NB2 (Processing Sequences Using RNNs), which splits
the CTA ridership series with date-string slicing on a DatetimeIndex:

    mulvar_train = mulvar_df["2016-01":"2018-12"]
    mulvar_valid = mulvar_df["2019-01":"2019-05"]
    mulvar_test  = mulvar_df["2019-06":]

Project spec fixes the test boundary at October 2022.  The train/val carve
is the last 3 months of the spec's train range, matching NB2's convention
of using the most recent slice for validation.

We split on `valid_time` (not init_time) because the spec's hard rule is
that test-period observations cannot influence training -- so the cutoff
applies to the timestamp the observation was MADE, not the timestamp the
forecast was issued.
"""

from pathlib import Path
import pandas as pd

STATIONS = ["20380357", "21609641"]

# Date-string slices, NB2 style.  Spec sets test = 2022-10 onward.
TRAIN_RANGE = ("2021-04", "2022-06")
VALID_RANGE = ("2022-07", "2022-09")
TEST_RANGE  = ("2022-10", "2023-04")


def split_station(parquet_path: Path):
    """Read one station's tidy parquet and slice it three ways."""
    df = pd.read_parquet(parquet_path)
    df = df.set_index("valid_time").sort_index()

    train = df.loc[TRAIN_RANGE[0]:TRAIN_RANGE[1]]
    valid = df.loc[VALID_RANGE[0]:VALID_RANGE[1]]
    test  = df.loc[TEST_RANGE[0]:TEST_RANGE[1]]
    return train, valid, test


def report(name: str, df: pd.DataFrame) -> None:
    if df.empty:
        print(f"  {name:6s}: EMPTY")
        return
    n_days = df.index.normalize().nunique()
    print(f"  {name:6s}: {len(df):>7,} rows | "
          f"{df.index.min()}  ->  {df.index.max()} | "
          f"{n_days} unique days")


def main() -> None:
    data_root = Path("data")
    for sid in STATIONS:
        in_path = data_root / sid / f"tidy_{sid}.parquet"
        print(f"\n{'='*70}\nstation {sid}\n{'='*70}")

        train, valid, test = split_station(in_path)
        report("train", train)
        report("valid", valid)
        report("test",  test)

        for name, part in [("train", train), ("valid", valid), ("test", test)]:
            out = data_root / sid / f"{name}_{sid}.parquet"
            part.reset_index().to_parquet(out)
        print(f"  wrote: train_/valid_/test_{sid}.parquet")


if __name__ == "__main__":
    main()



station 20380357
  train : 188,181 rows | 2021-04-21 01:00:00+00:00  ->  2022-06-30 23:00:00+00:00 | 436 unique days
  valid :  39,744 rows | 2022-07-01 00:00:00+00:00  ->  2022-09-30 23:00:00+00:00 | 92 unique days
  test  :  87,864 rows | 2022-10-01 00:00:00+00:00  ->  2023-04-22 17:00:00+00:00 | 204 unique days
  wrote: train_/valid_/test_20380357.parquet

station 21609641
  train : 188,181 rows | 2021-04-21 01:00:00+00:00  ->  2022-06-30 23:00:00+00:00 | 436 unique days
  valid :  39,744 rows | 2022-07-01 00:00:00+00:00  ->  2022-09-30 23:00:00+00:00 | 92 unique days
  test  :  87,864 rows | 2022-10-01 00:00:00+00:00  ->  2023-04-22 17:00:00+00:00 | 204 unique days
  wrote: train_/valid_/test_21609641.parquet


In [6]:
"""
Windowed tf.data datasets for the runoff project.

Pattern: NB2 (Géron, "Processing Sequences Using RNNs"), specifically the
high-level recipe:

    tf.keras.utils.timeseries_dataset_from_array(
        data, targets=series[sequence_length:],
        sequence_length=L, batch_size=B, shuffle=True, seed=42,
    )

Adaptations for this project (all from milestone1_notes / Han & Morrison):
  * L = 6  hours of history  (paper's best lag time)
  * H = 18 hours forecast horizon (NWM short-range range)
  * Multivariate input: [error, nwm_q, usgs_q] at lead_h=1
  * Seq2seq target: a vector of H future errors per window
  * Normalization fit on TRAIN ONLY (per spec's no-leakage rule)

Only one row per `valid_time` is kept (the lead_h=1 row), giving a clean
hourly grid -- this is the simplest NB2-compatible flattening of our
long-format tidy frame. Per-lead-time forecasts can be added later if we
want to match Han & Morrison's exact multi-feature-per-issue setup.

Outputs three tf.data.Datasets per station: train_ds, valid_ds, test_ds.
"""

from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

# Hyperparameters from milestone notes (Han & Morrison sensitivity analysis).
L = 6              # input window length (hours)
H = 18             # forecast horizon (hours)
BATCH_SIZE = 128

FEATURE_COLS = ["error", "nwm_q", "usgs_q"]
TARGET_COL = "error"

STATIONS = ["20380357", "21609641"]


# ---------- features -------------------------------------------------------- #

def prepare_features(split_df: pd.DataFrame) -> pd.DataFrame:
    """
    Long-format split -> hourly wide frame indexed by valid_time.

    Keeps only the lead_h=1 row per valid_time (one observation per hour),
    forward-fills any missing usgs_q (and the derived `error`) so the
    series is continuous for windowing.
    """
    df = split_df[split_df["lead_h"] == 1].copy()
    df = df.set_index("valid_time").sort_index()
    df = df[FEATURE_COLS]

    # Forward-fill the USGS gaps (~6% on station 2). Same semantics as
    # NB2's handling of intermittent missing values: keep cadence, accept
    # a small amount of repeated information rather than break the grid.
    df = df.ffill().bfill()
    return df


# ---------- normalization (fit on train only) ------------------------------ #

def fit_norm(train_features: pd.DataFrame):
    """Per-column mean/std from training features only."""
    mean = train_features.mean()
    std = train_features.std().replace(0.0, 1.0)
    return mean.values, std.values


def apply_norm(features: pd.DataFrame, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((features.values - mean) / std).astype(np.float32)


# ---------- seq2seq targets ----------------------------------------------- #

def build_seq2seq_targets(series: np.ndarray, L: int, H: int) -> np.ndarray:
    """
    Build 2D target array of shape (n_windows, H) where row i corresponds
    to the input window data[i:i+L] and contains the next H values:

        targets[i] = series[i+L : i+L+H]
    """
    N = len(series)
    n_windows = N - L - H + 1
    if n_windows <= 0:
        raise ValueError(f"Series too short: N={N}, need at least L+H={L+H}")
    out = np.empty((n_windows, H), dtype=np.float32)
    for h in range(H):
        out[:, h] = series[L + h : N - H + 1 + h]
    return out


# ---------- tf.data dataset ----------------------------------------------- #

def make_dataset(
    features: np.ndarray,
    targets: np.ndarray,
    L: int,
    H: int,
    batch_size: int,
    shuffle: bool,
    seed: int = 42,
) -> tf.data.Dataset:
    """
    NB2's high-level recipe: timeseries_dataset_from_array with explicit
    targets aligned to the END of each input window.

    `features[:-H]` truncates so the last window's 18-hour target lives
    inside the data. Windows are shuffled (per NB2: fine because we
    shuffle WHOLE windows, each of which preserves temporal order).
    """
    return tf.keras.utils.timeseries_dataset_from_array(
        data=features[:-H],
        targets=targets,
        sequence_length=L,
        batch_size=batch_size,
        shuffle=shuffle,
        seed=seed,
    )


# ---------- per-station pipeline ------------------------------------------ #

def build_station_datasets(station_id: str, data_root: Path = Path("data")):
    """Return (train_ds, valid_ds, test_ds, mean, std) for one station."""
    split_paths = {
        s: data_root / station_id / f"{s}_{station_id}.parquet"
        for s in ("train", "valid", "test")
    }
    splits = {s: pd.read_parquet(p) for s, p in split_paths.items()}
    feats = {s: prepare_features(df) for s, df in splits.items()}

    mean, std = fit_norm(feats["train"])

    datasets = {}
    for name, df in feats.items():
        x = apply_norm(df, mean, std)                      # (N, n_features)
        y_series = df[TARGET_COL].values.astype(np.float32)  # un-normalized error
        # Note: targets are RAW error in cms, not normalized -- so the model
        # learns to output cms directly. (Han & Morrison does this too.)
        targets = build_seq2seq_targets(y_series, L=L, H=H)
        ds = make_dataset(
            features=x,
            targets=targets,
            L=L,
            H=H,
            batch_size=BATCH_SIZE,
            shuffle=(name == "train"),
        )
        datasets[name] = ds

    return datasets, mean, std


# ---------- reporting ----------------------------------------------------- #

def report_station(station_id: str) -> None:
    print(f"\n{'='*70}\nstation {station_id}\n{'='*70}")
    datasets, mean, std = build_station_datasets(station_id)

    print(f"\n  hyperparameters:  L={L}  H={H}  batch={BATCH_SIZE}")
    print(f"  feature columns:  {FEATURE_COLS}")
    print(f"  target column:    {TARGET_COL}  (raw cms, not normalized)")
    print(f"\n  train norm stats (mean / std):")
    for col, m, s in zip(FEATURE_COLS, mean, std):
        print(f"    {col:8s}  mean={m:8.3f}   std={s:8.3f}")

    print(f"\n  dataset shapes:")
    for name, ds in datasets.items():
        x_batch, y_batch = next(iter(ds))
        n_batches = sum(1 for _ in ds)
        print(f"    {name:6s}  batches={n_batches:>4}   "
              f"x={tuple(x_batch.shape)}   y={tuple(y_batch.shape)}")


def main() -> None:
    for sid in STATIONS:
        report_station(sid)


if __name__ == "__main__":
    main()



station 20380357

  hyperparameters:  L=6  H=18  batch=128
  feature columns:  ['error', 'nwm_q', 'usgs_q']
  target column:    error  (raw cms, not normalized)

  train norm stats (mean / std):
    error     mean=  -4.183   std=   5.547
    nwm_q     mean=   4.452   std=   5.545
    usgs_q    mean=   0.270   std=   0.132

  dataset shapes:
    train   batches=  82   x=(128, 6, 3)   y=(128, 18)
    valid   batches=  18   x=(128, 6, 3)   y=(128, 18)
    test    batches=  38   x=(128, 6, 3)   y=(128, 18)

station 21609641

  hyperparameters:  L=6  H=18  batch=128
  feature columns:  ['error', 'nwm_q', 'usgs_q']
  target column:    error  (raw cms, not normalized)

  train norm stats (mean / std):
    error     mean=   0.212   std=   1.513
    nwm_q     mean=  13.158   std=  14.573
    usgs_q    mean=  13.374   std=  14.441

  dataset shapes:
    train   batches=  82   x=(128, 6, 3)   y=(128, 18)
    valid   batches=  18   x=(128, 6, 3)   y=(128, 18)
    test    batches=  38   x=(128, 6,

In [7]:
"""
Persistence baseline: use NWM forecast unchanged, i.e., predicted error = 0
at every lead time. The "thing to beat" per milestone Part H.

Saves predictions in the same shape as DL model predictions: (n_windows, H).
Output: data/<station>/preds_persistence_<station>.npy

Relies on names defined in the earlier windows cell:
  L, H, STATIONS, prepare_features
"""

from pathlib import Path
import numpy as np
import pandas as pd


def predict_persistence(station_id: str, data_root: Path = Path("data")) -> np.ndarray:
    test_split = pd.read_parquet(data_root / station_id / f"test_{station_id}.parquet")
    test_feats = prepare_features(test_split)
    n_windows = len(test_feats) - L - H + 1
    return np.zeros((n_windows, H), dtype=np.float32)


def main() -> None:
    data_root = Path("data")
    for sid in STATIONS:
        out_path = data_root / sid / f"preds_persistence_{sid}.npy"
        preds = predict_persistence(sid, data_root)
        np.save(out_path, preds)
        print(f"  station {sid}: saved {out_path.name}  shape={preds.shape}")


if __name__ == "__main__":
    main()

  station 20380357: saved preds_persistence_20380357.npy  shape=(4850, 18)
  station 21609641: saved preds_persistence_21609641.npy  shape=(4850, 18)


In [8]:
"""
DL training: GRU(32) and LSTM stacked, per station.

Architectures and hyperparameters from milestone Part H, traceable to NB2
and Han & Morrison:

  * Model 1 -- gru:           GRU(32) -> Dense(18)
                              NB2 model zoo entry 5 (multi-step Dense head).
  * Model 2 -- lstm_stacked:  LSTM(32, return_sequences=True) -> LSTM(32) -> Dense(18)
                              NB2 model zoo entry 3 (deep RNN), with LSTM cells.

Compile / train (milestone Part C.10 + Part H):
  * Loss: Huber (robust to flood-peak outliers)
  * Optimizer: Adam(lr=1e-3, clipnorm=1.0)
  * Metric: MAE
  * Callbacks: EarlyStopping(val_mae, patience=20, restore_best_weights),
               ReduceLROnPlateau(factor=0.5, patience=5)
  * fit_and_evaluate convention from NB2.
  * clear_session + set_seed(42) before each model build.

Saves per (station, model):
  data/<station>/<model>_<station>.keras       trained model
  data/<station>/preds_<model>_<station>.npy   test-set predictions, shape (n_windows, H)

Relies on names defined in earlier cells:
  L, H, FEATURE_COLS, STATIONS, build_station_datasets
"""

from pathlib import Path
import numpy as np
import tensorflow as tf

EPOCHS = 200
LEARNING_RATE = 1e-3
GRU_UNITS = 32       # paper's optimum
LSTM_UNITS = 32      # paper's optimum


# ---------- model builders ----------------------------------------------- #

def _seed_and_clear() -> None:
    tf.keras.backend.clear_session()
    tf.random.set_seed(42)


def build_gru(L: int, n_features: int, H: int) -> tf.keras.Model:
    """NB2 entry 5: single recurrent layer + multi-step Dense head."""
    _seed_and_clear()
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(L, n_features)),
        tf.keras.layers.GRU(GRU_UNITS),
        tf.keras.layers.Dense(H),
    ])


def build_lstm_stacked(L: int, n_features: int, H: int) -> tf.keras.Model:
    """NB2 entry 3 (deep RNN), with LSTM cells per milestone Part H."""
    _seed_and_clear()
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(L, n_features)),
        tf.keras.layers.LSTM(LSTM_UNITS, return_sequences=True),
        tf.keras.layers.LSTM(LSTM_UNITS),
        tf.keras.layers.Dense(H),
    ])


MODELS = {
    "gru": build_gru,
    "lstm_stacked": build_lstm_stacked,
}


# ---------- compile / callbacks / fit ------------------------------------ #

def compile_model(model: tf.keras.Model, lr: float = LEARNING_RATE) -> None:
    model.compile(
        loss=tf.keras.losses.Huber(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        metrics=["mae"],
    )


def make_callbacks() -> list:
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_mae", patience=20, restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_mae", factor=0.5, patience=5,
        ),
    ]


def fit_and_evaluate(model, train_ds, valid_ds, epochs=EPOCHS):
    history = model.fit(
        train_ds, validation_data=valid_ds, epochs=epochs,
        callbacks=make_callbacks(), verbose=2,
    )
    val_loss, val_mae = model.evaluate(valid_ds, verbose=0)
    return history, val_mae


# ---------- per (station, model) experiment ------------------------------ #

def train_one(station_id: str, model_name: str, data_root: Path = Path("data")):
    print(f"\n{'-'*70}\n  {station_id}  /  {model_name}\n{'-'*70}")

    datasets, _, _ = build_station_datasets(station_id, data_root)
    n_features = len(FEATURE_COLS)

    model = MODELS[model_name](L=L, n_features=n_features, H=H)
    compile_model(model)
    model.summary(line_length=70)

    _, val_mae = fit_and_evaluate(model, datasets["train"], datasets["valid"])
    test_loss, test_mae = model.evaluate(datasets["test"], verbose=0)

    model.save(data_root / station_id / f"{model_name}_{station_id}.keras")
    preds = model.predict(datasets["test"], verbose=0)
    np.save(data_root / station_id / f"preds_{model_name}_{station_id}.npy", preds)

    print(f"  val_mae:  {val_mae:.4f} cms")
    print(f"  test_mae: {test_mae:.4f} cms")
    return val_mae, test_mae


# ---------- main --------------------------------------------------------- #

def main() -> None:
    gpus = tf.config.list_physical_devices("GPU")
    print(f"GPUs visible: {len(gpus)}  ({'CPU only' if not gpus else gpus})")

    summary = {}
    for sid in STATIONS:
        print(f"\n{'='*70}\nstation {sid}\n{'='*70}")
        for model_name in MODELS:
            val_mae, test_mae = train_one(sid, model_name)
            summary[(sid, model_name)] = (val_mae, test_mae)

    print(f"\n{'='*70}\nfinal MAE summary (cms, lower=better)\n{'='*70}")
    print(f"  {'station':<12} {'model':<14} {'val_mae':>10} {'test_mae':>10}")
    for (sid, name), (vm, tm) in summary.items():
        print(f"  {sid:<12} {name:<14} {vm:>10.4f} {tm:>10.4f}")


if __name__ == "__main__":
    main()

GPUs visible: 1  ([PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')])

station 20380357

----------------------------------------------------------------------
  20380357  /  gru
----------------------------------------------------------------------


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Layer (type)                 ┃ Output Shape          ┃     Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ gru (GRU)                    │ (None, 32)            │       3,552 │
├──────────────────────────────┼───────────────────────┼─────────────┤
│ dense (Dense)                │ (None, 18)            │         594 │
└──────────────────────────────┴───────────────────────┴─────────────┘

 Total params: 4,146 (16.20 KB)

 Trainable params: 4,146 (16.20 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Layer (type)                 ┃ Output Shape          ┃     Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ lstm (LSTM)                  │ (None, 6, 32)         │       4,608 │
├──────────────────────────────┼───────────────────────┼─────────────┤
│ lstm_1 (LSTM)                │ (None, 32)            │       8,320 │
├──────────────────────────────┼───────────────────────┼─────────────┤
│ dense (Dense)                │ (None, 18)            │         594 │
└──────────────────────────────┴───────────────────────┴─────────────┘

 Total params: 13,522 (52.82 KB)

 Trainable params: 13,522 (52.82 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
82/82 - 4s - 48ms/step - loss: 0.3630 - mae: 0.6447 - val_loss: 0.0229 - val_mae: 0.1184 - learning_rate: 0.0010
Epoch 2/200
82/82 - 3s - 35ms/step - loss: 0.2499 - mae: 0.4377 - val_loss: 0.0217 - val_mae: 0.0983 - learning_rate: 0.0010
Epoch 3/200
82/82 - 2s - 22ms/step - loss: 0.2337 - mae: 0.4117 - val_loss: 0.0194 - val_mae: 0.0521 - learning_rate: 0.0010
Epoch 4/200
82/82 - 2s - 23ms/step - loss: 0.2236 - mae: 0.3961 - val_loss: 0.0204 - val_mae: 0.0742 - learning_rate: 0.0010
Epoch 5/200
82/82 - 2s - 22ms/step - loss: 0.2160 - mae: 0.3876 - val_loss: 0.0197 - val_mae: 0.0679 - learning_rate: 0.0010
Epoch 6/200
82/82 - 3s - 31ms/step - loss: 0.2110 - mae: 0.3810 - val_loss: 0.0195 - val_mae: 0.0554 - learning_rate: 0.0010
Epoch 7/200
82/82 - 2s - 22ms/step - loss: 0.2053 - mae: 0.3743 - val_loss: 0.0186 - val_mae: 0.0538 - learning_rate: 0.0010
Epoch 8/200
82/82 - 3s - 35ms/step - loss: 0.1994 - mae: 0.3655 - val_loss: 0.0178 - val_mae: 0.0518 - learning_rate: 0.0010
